# Model design

A *nnodely* model is a graph of **streams**: the signals read from data
(`Input`), the values computed from them by **layers** and arithmetic, and the
**outputs** you give a name to. A `Modely` turns that graph into a Keras model
that can be run, trained, validated and exported.

This tutorial builds a model step by step:

- how inputs are declared, and the shape convention every stream follows,
  `(S*, T, D*)`;
- how streams are combined with operators and layers, and how the shapes flow
  through them;
- how a `Modely` is declared, inspected, built and run;
- how a model is reused as a block of a bigger one (composition).

Every model goes through the same steps. This tutorial covers the first two;
the others go through the rest:

1. declare the streams: inputs, layers, outputs;
2. `Modely(name, inputs, outputs)`, its objectives with `minimize(...)`, then
   `build()`;
3. a `DataLoader` on the built model - [Data management](02_data_management.ipynb);
4. `train(...)` - [Model training](03_model_training.ipynb) - and `validate(...)`
   - [Model validation](04_model_validation.ipynb);
5. `save`, `load` and export - [Save, load and export](05_save_load_export.ipynb).

In [1]:
import os

# Pick the Keras backend before nnodely (or Keras) is imported:
# "tensorflow" (the default), "torch" or "jax".
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np

from nnodely import (
    Concatenate,
    Constant,
    Fir,
    Input,
    Linear,
    Modely,
    Output,
    Parameter,
    ReLU,
    Select,
    Tanh,
    TimeSelect,
    set_seed,
)

set_seed(42)  # the same weights, and the same results, on every run

## Inputs

An `Input` is a signal the data provides. It is declared once, with the shape
of its data, by three keyword arguments:

- `tw`, the **time window**: how many consecutive samples of the signal the
  model reads at once. `tw=5` is the last 5 samples, up to the current one;
  `tw=(past, future)` also reads samples after the current one, for example a
  target one step ahead. Without `tw` the input has no window: one value per
  sample.
- `dim`, the **features** of one sample: `dim=3` is a vector of 3 values,
  `dim=(2, 3)` a 2x3 matrix. One feature by default.
- `seq`, the **sequence**: several windows in a row, for example the steps of a
  trajectory. It needs a window, and is covered below.

In [2]:
speed = Input("speed", tw=5)  # the last 5 samples of a scalar signal
throttle = Input("throttle", tw=10)  # the last 10 samples
imu = Input("imu", dim=3, tw=4)  # 3 features per sample, 4 samples
mass = Input("mass")  # one value per sample, no window
road = Input("road", tw=(3, 2))  # 3 samples up to now, and the 2 after

for stream in (speed, throttle, imu, mass, road):
    print(f"{stream.name:9s} {stream.shape}")

speed     Shape(S=(), T=5, D=(1,))
throttle  Shape(S=(), T=10, D=(1,))
imu       Shape(S=(), T=4, D=(3,))
mass      Shape(S=(), T=None, D=(1,))
road      Shape(S=(), T=5, D=(1,))


## The shape convention: `(S*, T, D*)`

Every stream - an input, a layer's result, a parameter - has a shape made of
three groups of axes, printed `Shape(S=..., T=..., D=...)`:

| Group | Name | What it holds | Declared with |
|---|---|---|---|
| `S*` | sequence axes | zero or more axes of consecutive steps, outermost first: the steps of a trajectory, the rollout of a recurrent model | `seq=H` (`-1` for a length that follows the data) |
| `T` | time window | the consecutive samples of the signal read at once, oldest first; `None` when the stream has no window | `tw=n` or `tw=(past, future)` |
| `D*` | dims | the features of one sample: a scalar is `(1,)`, a vector `(3,)`, a matrix `(2, 3)` | `dim=` |

At run time a stream is a tensor laid out `(batch, *S, T, *D)`: the batch axis
first, then the sequence axes, the window, and the features last. A stream
without a window drops that axis: `(batch, *S, *D)`. The batch is never part of
the declared shape: it comes with the data.

The `*` marks the groups that can hold any number of axes: none, one or more
sequence axes and one or more feature axes, but always exactly one window (or
none).

In [3]:
trajectory = Input("trajectory", dim=2, seq=20, tw=1)  # 20 steps of a 2-D state
runs = Input("runs", dim=2, seq=(4, -1), tw=3)  # 4 runs of any length

for stream in (speed, imu, mass, trajectory, runs):
    shape = stream.shape
    print(
        f"{stream.name:10s} {str(shape):32s} seq axes {shape.seq_axes}, "
        f"window axis {shape.tw_axis}, dim axes {shape.dim_axes}"
    )

speed      Shape(S=(), T=5, D=(1,))         seq axes (), window axis 1, dim axes (2,)
imu        Shape(S=(), T=4, D=(3,))         seq axes (), window axis 1, dim axes (2,)
mass       Shape(S=(), T=None, D=(1,))      seq axes (), window axis None, dim axes (1,)
trajectory Shape(S=(20,), T=1, D=(2,))      seq axes (1,), window axis 2, dim axes (3,)
runs       Shape(S=(4, dyn), T=3, D=(2,))   seq axes (1, 2), window axis 3, dim axes (4,)


The tensor axes printed above are where each group sits in the array the model
reads: for `trajectory`, a batch of 8 samples is an array of shape
`(8, 20, 1, 2)`. A dynamic sequence (`-1`, printed `dyn`) takes its length from
the data.

A sequence is made of windows, so it needs one: `tw=1` for one sample per step.

In [4]:
try:
    Input("bad", seq=20)
except ValueError as error:
    print(error)

bad: a sequence needs a window: declare tw as well, tw=1 for one sample per step.


## Combining streams

Streams are combined with the Python operators `+ - * / ** %` - with each other
or with numbers - and with **layers**, which are configured first and then
called on streams. The result is a new stream, whose shape is computed when it
is declared: a wrong combination fails right away, not at training time.

The operators follow one broadcasting rule, the same for every stream:

- **window**: two streams must have the same window, or one of them has none,
  and then it applies to every sample of the other's;
- **sequence**: the same, or absent in one of them;
- **dims**: as NumPy broadcasts them, from the last axis (equal, or 1).

In [5]:
def show(**streams):
    """Print the shape of each stream, under the name it is given here."""
    for label, stream in streams.items():
        print(f"{label:28s} {stream.shape}")


show(
    speed_x2_plus_1=speed * 2.0 + 1.0,  # numbers have no window
    speed_times_mass=speed * mass,  # mass has none either
    imu_scaled=imu * Constant(value=[1.0, 0.5, 0.25]),  # one gain per feature
    imu_squared=imu**2,
)

speed_x2_plus_1              Shape(S=(), T=5, D=(1,))
speed_times_mass             Shape(S=(), T=5, D=(1,))
imu_scaled                   Shape(S=(), T=4, D=(3,))
imu_squared                  Shape(S=(), T=4, D=(3,))


A single sample is a window of 1 (`T=1`) and does not combine with a longer
window: subtracting the newest sample from a window of 5 is ambiguous. Select
the sample **without** its window, `keep_tw=False`, and it applies to each
sample of the window:

In [6]:
newest = TimeSelect(-1)(speed)  # T=1
try:
    speed - newest
except ValueError as error:
    print("ValueError:", error)

newest_value = TimeSelect(-1, keep_tw=False)(speed)  # no window
show(newest=newest, newest_value=newest_value, relative_speed=speed - newest_value)

ValueError: Subtract10: streams with windows of 1 and 5 cannot be combined: the windows must be the same, or a stream have none, which applies to every sample of the other's. A sample taken with TimeSelect(idx, keep_tw=False) has none.
newest                       Shape(S=(), T=1, D=(1,))
newest_value                 Shape(S=(), T=None, D=(1,))
relative_speed               Shape(S=(), T=5, D=(1,))


## Layers

A layer is configured first, then called on a stream (or a list of streams).
Each layer has a precise effect on the shape - these are the most common ones;
the later tutorials cover each family in depth:

| Layer | Acts on | Example |
|---|---|---|
| `Fir(features)` | the window: a weighted sum of its samples, one output sample | `T=10` -> `T=1` |
| `Linear(features)` | the dims of each sample, the window kept | `D=(3,)` -> `D=(8,)` |
| `Tanh()`, `ReLU()`, `Sin()`, ... | each value | unchanged |
| `Select(i)`, `Range(a, b)`, `Concatenate()` | the dims | pick, slice, join |
| `TimeSelect(i)`, `TimeRange(a, b)` | the window | pick, slice |

In [7]:
throttle_filtered = Fir(features=1)([throttle])  # 10 samples -> 1
imu_features = Linear(features=8)([imu])  # 3 features -> 8, per sample
imu_hidden = Tanh()(imu_features)  # elementwise
imu_summary = Fir(features=1)([Linear(features=1)([imu_hidden])])
joined = Concatenate()([throttle_filtered, imu_summary])
show(
    throttle_filtered=throttle_filtered,
    imu_features=imu_features,
    imu_hidden=imu_hidden,
    imu_summary=imu_summary,
    joined=joined,
    first_of_joined=Select(0)(joined),
)

throttle_filtered            Shape(S=(), T=1, D=(1,))
imu_features                 Shape(S=(), T=4, D=(8,))
imu_hidden                   Shape(S=(), T=4, D=(8,))
imu_summary                  Shape(S=(), T=1, D=(1,))
joined                       Shape(S=(), T=1, D=(2,))
first_of_joined              Shape(S=(), T=1, D=(1,))


A layer object holds its weights. Calling the **same** object on two streams
applies the same weights to both; two objects are two layers, each with weights
of its own, whatever their names:

In [8]:
shared = Fir(features=1, name="shared_filter")
left = Input("left", tw=10)
right = Input("right", tw=10)
both = shared([left]) + shared([right])  # one kernel, read by both calls

## Parameters and constants

A `Parameter` is a value the training learns - a mass, a gain, a coefficient -
and a `Constant` one it never changes. Both are streams, combined like any
other, and declared with the same `seq`, `tw` and `dim` as an input. They have
a tutorial of their own: [Parameter and Constant](06_parameter_constant.ipynb).

In [9]:
drag = Parameter("drag", value=0.05)  # learned, starts at 0.05
dt = Constant("dt", value=0.1)  # fixed
show(drag=drag, dt=dt, drag_force=drag * TimeSelect(-1)(speed) ** 2)

drag                         Shape(S=(), T=None, D=(1,))
dt                           Shape(S=(), T=None, D=(1,))
drag_force                   Shape(S=(), T=1, D=(1,))


## Declaring the model

An `Output` gives a stream a public name: the key the model returns it under.
A `Modely` is then declared with a name, the inputs it reads and its outputs.

Here a car's next speed is predicted from its last speeds and throttle commands:
a filter of each, minus a quadratic drag.

In [10]:
speed_now = TimeSelect(-1)(speed)
acceleration = (
    Fir(features=1, name="throttle_gain")([throttle]) - drag * speed_now * speed_now
)
next_speed = Output("next_speed", speed_now + dt * acceleration)
gain = Output("throttle_effect", Fir(features=1, name="effect")([throttle]))

car = Modely("car", inputs=[speed, throttle], outputs=[next_speed, gain])
car

┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ model      ┃ car                         ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ inputs     │ speed, throttle             │
│ outputs    │ next_speed, throttle_effect │
│ minimizers │ -                           │
└────────────┴─────────────────────────────┘

A model prints as a table of the names it reads and returns. Its parts are
listed by `inputs`, `outputs` and `parameters`, each printed as a table, and each
also a list: pick an item by its name or by its index, the first column of the
table.

In [11]:
car.inputs

┏━━━━━┳━━━━━━━━━━┳━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ idx ┃ input    ┃ seq ┃ tw ┃ dim  ┃ declared ┃ fed back with ┃ data     ┃ starts from ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 0   │ speed    │ ()  │ 5  │ (1,) │ yes      │ -             │ required │ -           │
│ 1   │ throttle │ ()  │ 10 │ (1,) │ yes      │ -             │ required │ -           │
└─────┴──────────┴─────┴────┴──────┴──────────┴───────────────┴──────────┴─────────────┘

In [12]:
car.outputs

┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ idx ┃ output          ┃ seq ┃ tw ┃ dim  ┃ stream ┃ feeds back ┃ minimized by ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 0   │ next_speed      │ ()  │ 1  │ (1,) │ Add29  │ -          │ -            │
│ 1   │ throttle_effect │ ()  │ 1  │ (1,) │ effect │ -          │ -            │
└─────┴─────────────────┴─────┴────┴──────┴────────┴────────────┴──────────────┘

In [13]:
car.parameters

┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┓
┃ idx ┃ parameter            ┃ seq ┃ tw   ┃ dim     ┃ values ┃ trainable ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━┩
│ 0   │ throttle_gain_kernel │ ()  │ None │ (10, 1) │ 10     │ yes       │
│ 1   │ throttle_gain_bias   │ ()  │ None │ (1,)    │ 1      │ yes       │
│ 2   │ drag                 │ ()  │ None │ (1,)    │ 1      │ yes       │
│ 3   │ effect_kernel        │ ()  │ None │ (10, 1) │ 10     │ yes       │
│ 4   │ effect_bias          │ ()  │ None │ (1,)    │ 1      │ yes       │
├─────┼──────────────────────┼─────┼──────┼─────────┼────────┼───────────┤
│     │ total                │     │      │         │ 23     │ 23        │
└─────┴──────────────────────┴─────┴──────┴─────────┴────────┴───────────┘

In [14]:
print(car.parameters["drag"].value)  # by name
print(car.outputs[0].name)  # by index

[0.05]
next_speed


## Building and running the model

`build()` creates the Keras model and its weights, and returns the model itself,
so `Modely(...).build()` chains. The graph is checked once more: two inputs or
outputs with the same name, for example, are refused here. Objectives
(`minimize`) are declared before the build - see
[Model training](03_model_training.ipynb).

In [15]:
car.build()
print("built:", car.built)
print("inference reads:", [node.name for node in car.inference_inputs])

built: True
inference reads: ['speed', 'throttle']


A built model is run on arrays laid out as its inputs are declared, **with a
batch axis**: `(batch, *S, T, *D)`. One sample is a batch of 1. The inputs can
be given

- as a dict keyed by input name (or by the `Input` itself),
- as a list, one array per declared input in their order,
- alone, when the model has one input.

The results are a dict of backend tensors, one per output;
`keras.ops.convert_to_numpy` turns them into NumPy arrays.

In [16]:
batch = 4
speed_data = np.linspace(10.0, 12.0, batch * 5, dtype="float32").reshape(batch, 5, 1)
throttle_data = np.full((batch, 10, 1), 0.3, dtype="float32")

result = car({"speed": speed_data, "throttle": throttle_data})
for name, value in result.items():
    print(name, keras.ops.convert_to_numpy(value).shape)

same = car([speed_data, throttle_data])  # a list, in the order of inputs
also = car({speed: speed_data, throttle: throttle_data})
print(
    np.allclose(
        keras.ops.convert_to_numpy(same["next_speed"]),
        keras.ops.convert_to_numpy(also["next_speed"]),
    )
)

next_speed (4, 1, 1)
throttle_effect (4, 1, 1)
True


The output `next_speed` is one sample per batch element: shape `(4, 1, 1)`,
batch 4, window 1, one feature - `Fir` turned the windows into one sample.

## Composition: models as blocks

A model is also a building block. Called on **streams** instead of arrays, it
returns streams of a bigger graph, with its weights shared between every place
it is called. A block can be designed and trained on its own, then reused.

Here a motor model maps a throttle window to a force; a two-motor vehicle uses
it once per motor:

In [17]:
command = Input("command", tw=10)
motor = Modely(
    "motor",
    inputs=[command],
    outputs=[Output("force", ReLU()(Fir(features=1)([command])))],
).build()

front, rear = Input("front", tw=10), Input("rear", tw=10)
total_force = motor([front]) + motor([rear])  # the same weights, twice
vehicle = Modely(
    "vehicle", inputs=[front, rear], outputs=[Output("total_force", total_force)]
).build()
vehicle.models

┏━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ idx ┃ model ┃ inputs  ┃ outputs ┃ parameters ┃ built ┃
┡━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ 0   │ motor │ command │ force   │ 2          │ yes   │
└─────┴───────┴─────────┴─────────┴────────────┴───────┘

A block keeps the shapes it was declared with: the streams given to it must have
the shape of its inputs. Nothing is silently reshaped:

In [18]:
try:
    motor([Input("short", tw=5)])
except ValueError as error:
    print(error)

Model 'motor' reads its input 'command' as Shape(S=(), T=10, D=(1,)), so used as a block it takes a stream of that shape for it - the same dim, sequence and window; 'short' is Shape(S=(), T=5, D=(1,)). A block keeps the shapes it was declared with: give it a stream of that shape, or declare the model on the shape it is used with.


The parameters of a composed model are listed per model, and a block can be
frozen by its name, so that training the bigger model leaves it as it is:

In [19]:
vehicle.freeze("motor")
vehicle.parameters

┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┓
┃ idx ┃ parameter    ┃ seq ┃ tw   ┃ dim     ┃ values ┃ trainable ┃
┡━━━━━┻━━━━━━━━━━━━━━┻━━━━━┻━━━━━━┻━━━━━━━━━┻━━━━━━━━┻━━━━━━━━━━━┩
│ motor                                                          │
├─────┬──────────────┬─────┬──────┬─────────┬────────┬───────────┤
│ 0   │ Fir31_kernel │ ()  │ None │ (10, 1) │ 10     │ no        │
│ 1   │ Fir31_bias   │ ()  │ None │ (1,)    │ 1      │ no        │
├─────┼──────────────┼─────┼──────┼─────────┼────────┼───────────┤
│     │ total        │     │      │         │ 11     │ 0         │
└─────┴──────────────┴─────┴──────┴─────────┴────────┴───────────┘

`flatten()` returns the same model with every block inlined, one graph of
layers only - useful to look at, or to export to tools that do not know blocks.

In [20]:
flat = vehicle.flatten().build()
data = {"front": np.ones((1, 10, 1), "float32"), "rear": np.ones((1, 10, 1), "float32")}
print(
    keras.ops.convert_to_numpy(vehicle(data)["total_force"]).ravel(),
    keras.ops.convert_to_numpy(flat(data)["total_force"]).ravel(),
)

[4.752013] [4.752013]


## Summary

| Step | Code |
|---|---|
| A signal from data | `Input("x", tw=10)`, `Input("v", dim=3, tw=1)`, `Input("m")`, `Input("x", seq=H, tw=1)` |
| Its shape | `x.shape` -> `Shape(S=..., T=..., D=...)`; tensor `(batch, *S, T, *D)` |
| Combine | `+ - * / ** %`, layers `Layer(settings)(stream)` |
| Learned / fixed values | `Parameter("k", value=1.0)`, `Constant("g", value=9.81)` |
| Name what the model returns | `Output("y", stream)` |
| Declare and build | `Modely("name", inputs=[...], outputs=[...]).build()` |
| Inspect | `model`, `model.inputs`, `model.outputs`, `model.parameters`, `model.models` |
| Run | `model({"x": array})`, arrays `(batch, *S, T, *D)` |
| Reuse | `block([stream])` inside another model, `model.freeze("block")`, `model.flatten()` |

Next: [Data management](02_data_management.ipynb) turns recorded signals into
the samples a model reads.